# 구매이력 표현의 N/V 조건부 저랭크 변환 M2 — Dunnhumby seed 42

자유 사용자 ID 임베딩을 제거하고, 고객이 구매한 상품의 학습 임베딩을 모은 표현을 N/V에 따라 제한적으로 변환합니다.

- 사용자 기본표현: 같은 이진 그래프에서 학습되는 아이템 ID 임베딩의 정규화된 1-hop 구매이력 집계 `H_u`
- 조건부 변환: `[I + 0.05(c_N A_N + c_V A_V)]H_u`, `A_N/A_V`는 각각 rank 4
- `c_N`, `c_V`: train에서 계산한 N/V 백분위에서 train 유효 사용자 평균을 뺀 고정값
- 자유 사용자 ID 임베딩·아이템 경제속성·별도 N/V 점수·외부 재정렬 없음
- 고정: binary graph, uniform negative sampling, plain BPR, 100 epoch, 하나의 optimizer
- 학습: 새 M2 한 개만 seed 42로 실행
- 비교: 동일 protocol과 입력 manifest로 저장된 M1@64 결과를 재사용

이 실행은 1~683일 학습, 684~690일 신규상품 평가의 빠른 역사적 개발구간 탐색입니다. 한 seed이므로 유의성을 주장하지 않으며, 양성이면 그때 구매이력-only·shuffled N/V 대조군을 추가합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'ac3aa545f183de5c257206b58b2d030d65a40cbd'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('코드 고정 완료:', REVIEWED_SHA)

In [ ]:
import json
import torch
from lightgcn_clv_history_conditioned_lowrank import (
    configure_history_conditioned_lowrank_run,
    preflight_summary,
    run_history_conditioned_lowrank_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_history_conditioned_lowrank_run(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_history_conditioned_lowrank_historical_screen_v1'
    ),
    baseline_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    ),
)
summary = preflight_summary(cfg)
assert summary['trained_models'] == ['m2_history_conditioned_lowrank_transform']
assert summary['controls_trained'] == []
assert summary['m2']['free_user_id_embedding'] is False
assert summary['m2']['user_base'] == 'normalized_purchase_history'
assert summary['m2']['transform_rank'] == 4
assert summary['m2']['rho'] == 0.05
assert summary['m2']['explicit_item_features'] is False
assert summary['fixed']['graph'] == 'binary'
assert summary['fixed']['negative_sampling'] == 'uniform'
assert summary['fixed']['one_training_loop_and_optimizer'] is True
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_history_conditioned_lowrank_screen(cfg)

In [ ]:
from IPython.display import display

comparison = result_df.attrs['comparison'].copy()
reading = dict(result_df.attrs['screening_reading'])
paths = dict(result_df.attrs['result_paths'])
display_df = result_df.copy()
display_df.attrs = {}

print('절대지표:')
display(display_df.sort_values('model_id'))

core_metrics = [
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'coverage@10', 'n_distinct@10', 'top10_share@10',
]
print('M1@64 대비 핵심 변화:')
display(comparison[comparison['metric'].isin(core_metrics)].sort_values('metric'))
print('탐색 판독:', reading)
print('결과 파일:', paths)